#### Model will be Trained on One Attribute for Selection

##### Imports

In [26]:
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score

import xgboost as xgb
from xgboost import XGBRegressor

import lightgbm as lgb
from lightgbm import LGBMRegressor

import catboost as cb
from catboost import CatBoostRegressor

##### Directories and File Locations

In [6]:
BASE_DIR = Path.cwd().parent
SPLITS_DIR = BASE_DIR / "data" / "splits"

In [7]:
train_df = pd.read_csv(SPLITS_DIR / "train.csv")
validate_df = pd.read_csv(SPLITS_DIR / "validation.csv")
test_df = pd.read_csv(SPLITS_DIR / "test.csv")

Features that were selected from the previous notebook

In [8]:
feature_cols = ['GP_base', 
                'MIN_base', 
                'FG_PCT_base', 
                'FG3M', 
                'FG3A', 
                'FG3_PCT', 
                'FTM', 
                'FTA', 
                'FT_PCT', 
                'OREB', 
                'DREB', 
                'REB', 
                'AST', 
                'TOV', 
                'STL', 
                'BLK', 
                'BLKA', 
                'PF', 
                'PFD', 
                'PTS', 
                'PLUS_MINUS', 
                'OFF_RATING', 
                'DEF_RATING', 
                'NET_RATING', 
                'AST_PCT', 
                'AST_TO', 
                'AST_RATIO', 
                'OREB_PCT', 
                'DREB_PCT', 
                'REB_PCT', 
                'E_TOV_PCT', 
                'EFG_PCT', 
                'TS_PCT', 
                'USG_PCT', 
                'PACE', 
                'PIE', 
                'POSS', 
                'FGM_PG', 
                'FGA_PG']

Labels

In [9]:
target_col = "threePointShot"

Create X and Y

In [10]:
X_train = train_df[feature_cols]
Y_train = train_df[target_col]

X_validate = validate_df[feature_cols]
Y_validate = validate_df[target_col]

X_test = test_df[feature_cols]
Y_test = test_df[target_col]

##### Scale Features

This is so that all the features are on similar scales

In [15]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_validate_scaled = scaler.transform(X_validate)

X_test_scaled = scaler.transform(X_test)

##### Train Model

I will be starting off with Ridge Regression, this is so our features don't get pushed all the way to 0 which would essentially allow us to target key features, if I were to use something like Lasso regression, it would select the most important features and that would be bad down the line once we start to model other labels which would then neglect the other features.

Also, it is easy to train and interpret as I'm just starting out and helps me provide a benchmark to when I use other models later on!

In [10]:
model = Ridge()

model.fit(X_train_scaled, Y_train)

,alpha,1.0
,fit_intercept,True
,copy_X,True
,max_iter,None
,tol,0.0001
,solver,'auto'
,positive,False
,random_state,None


Now we make predictions

In [17]:
val_preds = model.predict(X_validate_scaled)

Calculate MAE

In [19]:
mae = mean_absolute_error(Y_validate, val_preds)

print(f"MAE: {mae:.2f}")

MAE: 4.88


Get R^2 Score

In [21]:
r2 = r2_score(Y_validate, val_preds)

print(f"R²: {r2:.3f}")

R²: 0.618


According to GPT, I should inspect the coefficients

In [22]:
coef_df = pd.DataFrame({
    "Feature": feature_cols,
    "Coefficient": model.coef_
})

coef_df.sort_values(
    by="Coefficient",
    ascending=False
).head(15)

,Feature,Coefficient
32,TS_PCT,4.939053
37,FGM_PG,3.560896
35,PIE,3.453941
19,PTS,3.443301
5,FG3_PCT,2.944187
27,OREB_PCT,2.270338
10,DREB,1.994315
36,POSS,1.536966
11,REB,1.182830
17,PF,1.132359


In [23]:
coef_df.sort_values(
    by="Coefficient",
    ascending=True
).head(15)

,Feature,Coefficient
2,FG_PCT_base,-11.046351
9,OREB,-4.953482
28,DREB_PCT,-4.468224
38,FGA_PG,-3.030653
7,FTA,-2.669179
4,FG3A,-2.308243
12,AST,-2.287855
15,BLK,-1.448751
0,GP_base,-0.769788
23,NET_RATING,-0.691778


##### Learnings

- So overall, this was a decent first model. 

- From the MAE = 4.88, this meant that for every players 3 point shot attribute, my model was off by about 5 points so if a player had a 80 rating, it would give it a 75 rating.

- The R^2 was 0.618 so the player stats roughly explained about 62% of the three point attribute.

- By inspecting the coefficients and doing some learning, things like FG3_PCT, PTS, and FGM_PG were highly postively correlating which made sense as those directly affect why someone might be considered a good shooter. The ones to stand out were FG3A and FG_PCT_base as they were both negative and at first it seemed like a good shooter should have these to be positive, however, there are other shots too that usually big men make so the model learned that.

##### XG Boost

With the help of GPT, I'll be implementing XGBoost here as it is supposedly better since basketball stats are never linear and a tree based regression system helps to model that more accurately.

In [11]:
# parameters provided to me, I have no idea what they actually mean
xgb_model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    random_state=42
)

xgb_model.fit(X_train, Y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


Predict

In [13]:
xgb_preds = xgb_model.predict(X_validate)

Calculate MAE and R^2

In [14]:
xgb_mae = mean_absolute_error(
    Y_validate,
    xgb_preds
)

xgb_r2 = r2_score(
    Y_validate,
    xgb_preds
)

print(f"MAE: {xgb_mae:.2f}")

print(f"R²: {xgb_r2:.3f}")

MAE: 3.18
R²: 0.750


##### Key Takeaways from XGBoost

- This model seemed to perform way better than the last one. 
- It doesn't need data normalization
- I'm going to try out two more models to see if they work any better

##### LightGBM

In [23]:
lgr = LGBMRegressor()
lgr.fit(X_train, Y_train)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000479 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5153
[LightGBM] [Info] Number of data points in the train set: 836, number of used features: 39
[LightGBM] [Info] Start training from score 74.162679


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


Predict

In [24]:
lgr_preds = lgr.predict(X_validate)

Calculate MAE and R^2

In [25]:
lgr_mae = mean_absolute_error(
    Y_validate,
    lgr_preds
)

lgr_r2 = r2_score(
    Y_validate,
    lgr_preds
)

print(f"MAE: {lgr_mae:.2f}")

print(f"R²: {lgr_r2:.3f}")

MAE: 3.28
R²: 0.764


##### CatBoost

In [30]:
cbr = CatBoostRegressor(
    iterations=2000,
    depth=6,
    learning_rate=0.03,
    loss_function='RMSE',
    random_state=42,
    verbose=100
)

cbr.fit(
    X_train,
    Y_train,
    eval_set=(X_validate, Y_validate),
    use_best_model=True
)

0:	learn: 12.0265889	test: 11.8827101	best: 11.8827101 (0)	total: 3.36ms	remaining: 6.72s
100:	learn: 4.6969796	test: 5.8131944	best: 5.8131944 (100)	total: 86ms	remaining: 1.62s
200:	learn: 3.5924305	test: 5.5679805	best: 5.5679375 (199)	total: 167ms	remaining: 1.5s
300:	learn: 2.8730817	test: 5.5142858	best: 5.5138774 (297)	total: 248ms	remaining: 1.4s
400:	learn: 2.2957656	test: 5.4800915	best: 5.4800915 (400)	total: 330ms	remaining: 1.31s
500:	learn: 1.9319510	test: 5.4543344	best: 5.4543344 (500)	total: 407ms	remaining: 1.22s
600:	learn: 1.6254788	test: 5.4316930	best: 5.4291841 (592)	total: 494ms	remaining: 1.15s
700:	learn: 1.4020053	test: 5.4311743	best: 5.4271221 (678)	total: 582ms	remaining: 1.08s
800:	learn: 1.2008782	test: 5.4281557	best: 5.4270487 (778)	total: 664ms	remaining: 994ms
900:	learn: 1.0364934	test: 5.4389792	best: 5.4270487 (778)	total: 749ms	remaining: 914ms
1000:	learn: 0.8991934	test: 5.4382699	best: 5.4270487 (778)	total: 842ms	remaining: 840ms
1100:	learn:

CatBoostRegressor(depth=6, iterations=2000, learning_rate=0.03, loss_function='RMSE', random_state=42, verbose=100)

Predict

In [31]:
cat_preds = cbr.predict(X_validate)

Calculate MAE and R^2

In [32]:
cat_mae = mean_absolute_error(
    Y_validate,
    cat_preds
)

cat_r2 = r2_score(
    Y_validate,
    cat_preds
)

print(f"MAE: {cat_mae:.2f}")

print(f"R²: {cat_r2:.3f}")

MAE: 3.00
R²: 0.798
